Randomised Riemannian Hamiltonian Monte Carlo for Sampling on $\mathbb{V}_{d,p}$

Defined by $g(X) = X^{T}X - \mathbb{I}_{p} = 0$, we have $\mathcal{M} := \{x \in \mathbb{R}^{d \times p} \mid g(x) = 0\}$

$T_{X}\mathcal{M} =  \{V \in \mathbb{R}^{d \times p} \mid V^{T}X + X^{T}V = 0 \}$

For $A \in \mathbb{R}^{d \times p}$ with $v_{i} \in \mathbb{R}^{d}$, $i = 1,...,p$ being the column vectors. We define $g_{ij}(A) = v_{i} \cdot v_{j} - \delta_{ij} = 0$


1) Dynamics

We are sampling from the von Mises-Fisher distribution which has density given by 

$p_{vMF}(X) \propto \exp{(\langle f_{1},x_{1} \rangle + ... + \langle f_{p},x_{p}\rangle)}$

In [ ]:
import numba
import numpy as np
from numpy.linalg import inv
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import time
import random

#p<=d
p=3
d=18
dimension = int(d*p)
num_of_constraints  = int(p*(p+1)/2)

#Matrix in distribution

F = np.eye(d,p)

F[3:6] = np.array([[0,2,-45],[-2,0,-4],[45,4,0]])

F[6:9] = np.eye(3,3)

F[9:12] = np.array([[0,2,-45],[-2,0,-4],[45,4,0]])

F[12:15] = np.eye(3,3)

F[15:18] = np.array([[0,2,-45],[-2,0,-4],[45,4,0]])
#K = 10 #Constant inversely proportional to variance.
#10 gets a very good curve, now to try 1!

K = 1.

@numba.jit(nopython=True)
def vec_to_matrix(q):
    X = np.zeros((d,p))
    for i in range(d):
        for j in range(p):
            X[i,j] = q[j*d+i]
    return X

@numba.jit(nopython=True)
def matrix_to_vec(X):
    #initialising filler array
    x = np.zeros(d*p)
    
    for i in range(d*p):
        i_index = i%d
        j_index =  int((i - i_index)/d)
        x[i] = X[i_index,j_index]
    return x

@numba.jit(nopython=True)
def dot_product(v1,v2):
    dot = 0
    for i in range(len(v1)):
        
        dot += v1[i]*v2[i]
        
    return dot

@numba.jit(nopython=True)
def matmul(matrix1,matrix2):
    a = matrix1.shape[0]
    b = matrix2.shape[1]
    c = matrix2.shape[0]
    rmatrix = np.zeros((a,b))
    for i in range(a):
        for j in range(b):
            for k in range(c):
                rmatrix[i,j] += matrix1[i,k] * matrix2[k,j]
    return rmatrix

@numba.jit(nopython=True)
def matrix_vec_multiplication(A,x):
    v = np.zeros(len(A))
    
    for i in range(len(A)):
        for j in range(len(x)):
                v[i] += A[i][j] * x[j]
    return v


@numba.jit(nopython=True)
def g_ij(q,i,j):
    
    X = vec_to_matrix(q)
    
    if i==j:
        y = np.linalg.norm(X[:,i])**2 - 1
    else:
        y = dot_product(X[:,i],X[:,j])
        
    return y

@numba.jit(nopython=True)
def G_matrix_input(X): #considering i<j.
    
    
    z = np.zeros((dimension,num_of_constraints))
    
    for i in range(p): #block diagonals
        z[d*i:d*(i+1),int(p*i-0.5*i*(i-1)):int(p*(i+1) - 0.5*i*(i+1))] = X[:,i:]
    
        #vector diagonals
        for j in range(p-i):
            z[(j+i)*d:(j+i+1)*d,int(p*i-0.5*i*(i-1) + j)] += X[:,i]
    z = z.T    
    return z

@numba.jit(nopython=True)
def G(q): #considering i<j.
    
    
    
    X = vec_to_matrix(q)
    
    z = np.zeros((dimension,num_of_constraints))
    
    for i in range(p): #block diagonals
        z[d*i:d*(i+1),int(p*i-0.5*i*(i-1)):int(p*(i+1) - 0.5*i*(i+1))] = X[:,i:]
    
        #vector diagonals
        for j in range(p-i):
            z[(j+i)*d:(j+i+1)*d,int(p*i-0.5*i*(i-1) + j)] += X[:,i]  
    z = z.T #could implement this above
    return z

Fvec = matrix_to_vec(F)

@numba.jit(nopython=True)
def potential_derv(q):
    mu = -K*Fvec
    return mu

#RATTLE Hamiltonian Flow
@numba.jit(nopython=True)
def RATTLE_with_Potential(x0,v0,t,dt,max_elim_iters):
    n = np.floor(t/dt)
    vn = v0
    qn = x0
    vhalf = v0
    G_q = G(qn)
    
    #Gram Matrix is GG^T
    gram = matmul(G_q,G_q.T)
    gram_inv = np.linalg.inv(gram)
    
    pderv = potential_derv(qn)
    
    residual_list = np.zeros(num_of_constraints)
    for i in range(int(n)):
        
        
        #solver for Lagrange position multipliers
        Q = qn + vn*dt - 0.5*dt*dt*pderv
        
        #non-linear gaussian elimination
        for k in range(max_elim_iters): #i>j
            for i in range(p):
                for j in range(i,p):
                    g_Q = g_ij(Q,i,j)
                    index = int(i*p - 0.5*i*(i-1) + j-i)
                    
                    residual_list[index] = g_Q
                    if abs(g_Q) < 1e-8:
                        continue
                    G_Q = G(Q)
                    
                    #should be sum of i's and js in indexing below
                    dlambda = g_Q/dot_product(G_Q[index,:],G_q[index,:])
                    Q = Q - G_q[index,:]*dlambda
            #break condition
            if np.all(np.abs(residual_list)<1e-8):
                break

        
        #half step
        vhalf = (Q-qn)/dt
        qn = Q
        
        pderv =  potential_derv(qn)
        G_q = G(qn)
        
        gram = matmul(G_q,G_q.T)
        gram_inv = np.linalg.inv(gram)
        
        #linear solver Lagrange velocity multipliers
        b = matrix_vec_multiplication(G_q,2*vhalf/dt - pderv)
        coeffs_v = matrix_vec_multiplication(gram_inv,b)
        
        #full step
        vn = vhalf - 0.5*dt*pderv - 0.5*dt*matrix_vec_multiplication(G_q.T,coeffs_v)
       
    return qn,vn

In [ ]:
print(F)

# Checks

In [ ]:
X = np.array([[1,2,3], [4,5,6],[7,8,9]])
#print(X[:,1:])
#g_ij checked
#G checked for what I want.
#print(X.T,X)
M = np.eye(d, p)
M = matrix_to_vec(M)
gram = matmul(G(M),G(M).T)
"G matrix should be #constraints by dimension 0.5*p*(p+1) by p*d" 
#6 by 9
print(gram)
M = np.eye(d, p)
print(matmul(G_matrix_input(M),G_matrix_input(M).T))


2) Event Time Sampling


In [ ]:
#Sampling Event Times
@numba.jit(nopython=True)
def time_exp(lam):
    t = np.random.exponential(lam)
    return t

3) Gaussian Sampling on Tangent Space

In [ ]:
@numba.jit(nopython=True)
def tangent_space_gaussian(q):
    
    
    v = np.random.normal(0.,1.0,dimension).T
    
    G_q = G(q)
    
    
    gram = matmul(G_q,G_q.T)
    gram_inv = np.linalg.inv(gram)
    
    proj_matrix = np.eye(dimension) - matmul(G_q.T,matmul(gram_inv,G_q))
    
    #sample 3d gaussian and then project onto tangent space.
    v = matrix_vec_multiplication(proj_matrix,v)
    
    return v

#check
X = np.eye(d,p)
X_vec = matrix_to_vec(X)
Z = tangent_space_gaussian(X_vec)
T_X = vec_to_matrix(Z)  

print(matmul(T_X.T,X) + matmul(X.T,T_X))

In [ ]:
@numba.jit(nopython=True)
def f(q):
    
    z = -K*dot_product(Fvec,q)
    
    return z

@numba.jit(nopython=True)
def hamiltonian(x,v):
    return f(x) + 0.5*dot_product(v,v)

4) Simulation

In [ ]:
#Initialise
T = 0.1
num_of_events = 1000
dt = 0.001

In [ ]:
@numba.jit(nopython=True)
def RRHMC(num_of_events,N,T):
    
    #Exponential Expected Value
    rate = T
    
    #initialisation of x on V_{d,p}
    x = matrix_to_vec(np.eye(d, p))
    v = tangent_space_gaussian(x)
    
    position_list = [x]
    
    
    for i in range(num_of_events):
        
        dt = time_exp(rate/N)
        t = dt*N
        
        h = hamiltonian(x,v)
    
        xnew,vnew = RATTLE_with_Potential(x,v,t,dt,50)
        
        h_new = hamiltonian(xnew,vnew)
        
        #metropolis hasting step
        u = np.random.rand()
        if u <= np.exp(-h_new+h):
            x = xnew
            

        
        position_list.append(x)
        
        v = tangent_space_gaussian(x)
        if i%10000==0:
            print(i)
        
        
    return position_list

In [ ]:
@numba.jit(nopython=True)
def RHMC(num_of_events,dt,T):
    
    #initialisation of x on V_{d,p}
    x = matrix_to_vec(np.eye(d, p))
    v = tangent_space_gaussian(x)
    
    position_list = [x]
    
    
    for i in range(num_of_events):
        
        h = hamiltonian(x,v)
        
        xnew,vnew = RATTLE_with_Potential(x,v,T,dt,50)
        
        h_new = hamiltonian(xnew,vnew)
        
        #metropolis hasting step
        u = np.random.rand()
        if u <= np.exp(-h_new+h):
            x = xnew
        
        position_list.append(x)
        v = tangent_space_gaussian(x)  
        if i%10000==0:
            print(i)
            
    return position_list

In [ ]:
t = time.time()
position = RRHMC(num_of_events,T/dt,T)
elapsed = time.time() - t
print('RT time =',elapsed)
t = time.time()
position = RHMC(num_of_events,dt,T)
elapsed = time.time() - t
print('DT time =',elapsed)
position_matrix = []
for i in position:
    position_matrix.append(vec_to_matrix(i))

# Check

In [ ]:
constraint_list = []

for i in position_matrix:
    constraint_list.append(np.linalg.norm(np.matmul(i.T,i)-np.eye(p)))
plt.plot(constraint_list)
print('Maximum Constraint Error =',max(constraint_list))

In [ ]:
import statsmodels.api as sm
T_list = np.linspace(0.001,1,500)
T_list = T_list[417:]
print(T_list)
number = len(T_list)
RMHMC_list = []
RT_RMHMC_list = []

RMHMC_accept = []
RT_RMHMC_accept = []

for i in T_list:
    #Initialise
    T = i
    num_of_events = 100000
    dt = 0.001
    print('WE ARE AT STAGE =',i)

    
    #Samples
    pos = RRHMC(num_of_events,T/dt,T)
        
    GMC_samples = RHMC(num_of_events,dt,T)
    
    
    
    #IAC GMC
    front = len(GMC_samples)//10
    N = len(GMC_samples)- front

    auto_corr_vector_GMC = []
    for i in GMC_samples:
        auto_corr_vector_GMC.append(f(i))
    
    auto_GM = sm.tsa.acf(auto_corr_vector_GMC[front:],nlags = N//50)
    
    IAC_GM = np.sum(auto_GM)
    
    RMHMC_list.append(IAC_GM)
    
    #IAC RT-RMHMC
    front_RM = len(pos)//10

    N = len(pos)-front_RM
    auto_corr_vector_RRHMC = []

    for i in pos:
           auto_corr_vector_RRHMC.append(f(i))
            
    auto_RM = sm.tsa.acf(auto_corr_vector_RRHMC[front_RM:],nlags = N//50)
    
    IAC_RT = np.sum(auto_RM)
    
    RT_RMHMC_list.append(IAC_RT)
    
    print('===================================================================')
    print(RT_RMHMC_list)
    print('===================================================================')
    
    print('===================================================================')
    print(RMHMC_list)
    print('===================================================================')
    
    plt.title('IAC for varying $\lambda$')
    plt.plot(np.array(T_list)[:len(RT_RMHMC_list)],2*np.array(RT_RMHMC_list)-1,'x-',label='RT-RMHMC')
    plt.plot(np.array(T_list)[:len(RT_RMHMC_list)],2*np.array(RMHMC_list)-1,'x-',label='RMHMC')
    plt.plot(np.array(T_list)[:len(RT_RMHMC_list)], np.ones(len(T_list[:len(RT_RMHMC_list)])),'--')
    plt.xlabel('Mean duration $\lambda^{-1}$')
    plt.ylabel('IAC')
    plt.yscale('log')
    plt.grid()
    plt.legend()
    plt.show()